In [1]:
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    IntegerType, LongType, DoubleType, TimestampType
)

# In Fabric, use the default Lakehouse Files/ path via relative POSIX-style paths.
# The /pvsyst/Files mount is not writable from the driver filesystem, which caused PermissionError.
LAKEHOUSE_FILES = "Files"

# We do NOT need to create these folders via local OS; use notebookutils.fs which
# manipulates the Lakehouse-backed file system instead of the container root.
for sub in ["output", "configs"]:
    # This creates (if needed) `Files/output` and `Files/configs` in the default Lakehouse
    notebookutils.fs.mkdirs(f"{LAKEHOUSE_FILES}/{sub}")

print("Files/ structure ready (no raw/ — comes from SharePoint shortcut)")

In [2]:
# Gold: pr_results
pr_results_schema = StructType([
    StructField('run_id',           StringType(),    False),
    StructField('plant_code',       StringType(),    False),
    StructField('plant_name',       StringType(),    True),
    StructField('period_start',     StringType(),    True),
    StructField('period_end',       StringType(),    True),
    StructField('resolution',       StringType(),    True),
    StructField('passed',           BooleanType(),   True),
    StructField('measured_kwh',     DoubleType(),    True),
    StructField('guaranteed_kwh',   DoubleType(),    True),
    StructField('expected_kwh',     DoubleType(),    True),
    StructField('delta_kwh',        DoubleType(),    True),
    StructField('delta_pct',        DoubleType(),    True),
    StructField('valid_days',       IntegerType(),   True),
    StructField('total_intervals',  IntegerType(),   True),
    StructField('valid_intervals',  IntegerType(),   True),
    StructField('c1_failed',        IntegerType(),   True),
    StructField('c2_failed',        IntegerType(),   True),
    StructField('c3_failed',        IntegerType(),   True),
    StructField('c4_failed',        IntegerType(),   True),
    StructField('c5_failed',        IntegerType(),   True),
    StructField('c6_failed',        IntegerType(),   True),
    StructField('aggregated_valid_hours', DoubleType(), True),
    StructField('daily_json',       StringType(),    True),
    StructField('report_html_path', StringType(),    True),
    StructField('report_pdf_path',  StringType(),    True),
    StructField('executed_at',      TimestampType(), True),
])
spark.createDataFrame([], pr_results_schema).write.format('delta').mode('overwrite').saveAsTable('pr_results')
print('pr_results ready')

In [3]:
# Gold: plants dimension — start with Plaza II only
plants_data = [
    ('PL2', 'Plaza II', 16800, 18310, 0.98),
]
plants_schema = StructType([
    StructField('plant_code',     StringType(),  False),
    StructField('plant_name',     StringType(),  False),
    StructField('ac_capacity_kw', IntegerType(), True),
    StructField('dc_capacity_kw', IntegerType(), True),
    StructField('guaranteed_epi', DoubleType(),  True),
])
spark.createDataFrame(plants_data, plants_schema).write.format('delta').mode('overwrite').saveAsTable('plants')
print(f'plants ready ({len(plants_data)} row)')

In [5]:
# silver: scada_raw — partitioned by plant_code
# Ingesta cruda de SCADA (renombrada + tipada). Lo escribe `scada_ingest`.
scada_raw_schema = StructType([
    StructField('plant_code',   StringType(),    False),
    StructField('Date',         TimestampType(), False),
    StructField('GHI_01',       DoubleType(),    True),
    StructField('POA_01',       DoubleType(),    True),
    StructField('POA_02',       DoubleType(),    True),
    StructField('POA_03',       DoubleType(),    True),
    StructField('REF_01',       DoubleType(),    True),
    StructField('REF_02',       DoubleType(),    True),
    StructField('WS_01',        DoubleType(),    True),
    StructField('T_AMB_01',     DoubleType(),    True),
    StructField('E_GRID_01',    DoubleType(),    True),
    StructField('SETPOINT_01',  DoubleType(),    True),
    StructField('source_file',  StringType(),    True),
    StructField('ingested_at',  TimestampType(), True),
])
spark.createDataFrame([], scada_raw_schema).write.format('delta').partitionBy('plant_code').mode('overwrite').saveAsTable('scada_raw')
print('scada_raw ready (partitioned by plant_code)')


# silver/gold: scada_proc — partitioned by plant_code
# NO se predefine schema completo: `scada_proc.Notebook` lo crea con mergeSchema=true
# porque las columnas calculadas dependen del nº de sensores POA/REF por planta.
# Aquí solo creamos la tabla vacía con las columnas mínimas garantizadas.
scada_proc_schema = StructType([
    StructField('plant_code',         StringType(),    False),
    StructField('Date',               TimestampType(), False),
    StructField('POA_avg_filt',       DoubleType(),    True),
    StructField('ALB_avg_filt',       DoubleType(),    True),
    StructField('poa_above_thr',      IntegerType(),   True),
    StructField('daily_hours_above',  DoubleType(),    True),
    StructField('c1',                 IntegerType(),   True),
    StructField('c2',                 IntegerType(),   True),
    StructField('c3',                 IntegerType(),   True),
    StructField('c4',                 IntegerType(),   True),
    StructField('hours_aggregated',   DoubleType(),    True),
    StructField('all_valid',          IntegerType(),   True),
    StructField('processed_at',       TimestampType(), True),
])
spark.createDataFrame([], scada_proc_schema).write.format('delta').partitionBy('plant_code').mode('overwrite').saveAsTable('scada_proc')
print('scada_proc ready (partitioned by plant_code; schema evoluciona con mergeSchema)')

In [6]:
# proc: proc_pvsyst — partitioned by plant_code
proc_pvsyst_schema = StructType([
    StructField('plant_code',     StringType(),    False),
    StructField('Date',           TimestampType(), False),
    StructField('GHI',            DoubleType(),    True),
    StructField('T_AMB',          DoubleType(),    True),
    StructField('ALB',            DoubleType(),    True),
    StructField('POA',            DoubleType(),    True),
    StructField('E_Grid',         DoubleType(),    True),
    StructField('source_file',    StringType(),    True),
    StructField('ingested_at',    TimestampType(), True),
])
spark.createDataFrame([], proc_pvsyst_schema).write.format('delta').partitionBy('plant_code').mode('overwrite').saveAsTable('proc_pvsyst')
print('proc_pvsyst ready')

In [7]:
# Catalog: ingested_files — idempotency guard
ingested_files_schema = StructType([
    StructField('plant_code',  StringType(),    False),
    StructField('file_path',   StringType(),    False),
    StructField('file_kind',   StringType(),    False),
    StructField('file_size',   LongType(),      True),
    StructField('row_count',   LongType(),      True),
    StructField('period_start',TimestampType(), True),
    StructField('period_end',  TimestampType(), True),
    StructField('resolution',  StringType(),    True),
    StructField('status',      StringType(),    True),
    StructField('error',       StringType(),    True),
    StructField('ingested_at', TimestampType(), True),
])
spark.createDataFrame([], ingested_files_schema).write.format('delta').mode('overwrite').saveAsTable('ingested_files')
print('ingested_files ready')
print('\nAll bootstrap tables created. Upload YAML configs to Files/configs/ next.')

In [8]:
spark.table('gold.plaza2_15min').show(3)
